# The Safe Resource Vault — Context Managers

Difficulty: Beginner | ~25 min | Requires basic Python (functions, classes, try/except)

In this notebook you'll build a simulated database resource manager that *guarantees*
cleanup even when code crashes mid-operation — then turn it into a commit-on-success,
roll-back-on-error transaction.

In [1]:
!pip install tabulate==0.10.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Section 0 — Warm up: the `with` statement

`with` guarantees cleanup: when the block ends — normally or with an error — Python
calls a special exit method for you. Run the file demo, then the tiny protocol demo.

In [2]:
# A real file: the with statement closes it automatically when the block ends.
with open("vault_notes.txt", "w", encoding="utf-8") as note_file:
    note_file.write("The vault always closes what it opens.\n")

print("After the with block, is the file closed?", note_file.closed)

After the with block, is the file closed? True


In [3]:
# __enter__ runs at the start of the with block, __exit__ at the end — always.
class TinyVault:
    def __enter__(self):
        print("entering")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("exiting")
        # return False lets exceptions propagate to the caller.
        return False

with TinyVault():
    print("inside the block")

entering
inside the block
exiting


## Section 1 — The scenario

Every request opens a database connection and runs a transaction that must either all
succeed or all be undone. Context managers promise cleanup no matter what happens inside
the block.

## Section 2 — The protocol: `__enter__` and `__exit__`

A context manager is any object with two methods. `__enter__` runs on entry; `__exit__`
runs when the block ends — success or error — and receives the exception info (or three
`None`s). You watched this live in the warm-up; now give it a real job.

## Section 3 — Part 1: a class-based connection manager

Write the two methods on a `DatabaseConnection` class. `__enter__` "opens" the connection
and returns it; `__exit__` "closes" it. Watch the three lines print in order.

In [4]:
# Part 1 — a class-based context manager.
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name

    def __enter__(self):
        print(f"Opening {self.db_name} connection...")
        return self  # `as connection` binds this

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Closing {self.db_name} connection...")
        return False  # re-raise errors from the block

with DatabaseConnection("users") as connection:
    print("Running a query inside the block...")

Opening users connection...
Running a query inside the block...
Closing users connection...


## Section 4 — Part 1: prove cleanup on error

Crash *inside* the block. Watch the order: `Closing connection...` prints before
`Caught outside the block:` — the connection closed even though the query never finished.

In [5]:
# Crash inside the block: __exit__ still runs.
try:
    with DatabaseConnection("users") as connection:
        raise ValueError("query timed out mid-request")
except ValueError as error:
    print("Caught outside the block:", error)

Opening users connection...
Closing users connection...
Caught outside the block: query timed out mid-request


## Section 5 — Part 2: the `@contextlib.contextmanager` shortcut

*Preview: We're using a decorator here to create a context manager in a shorter form.
You don't need to understand how decorators work yet; Lab 4 covers decorators in
detail.*

Same manager as a one-function generator. Code before `yield` runs on entry, code after
runs on exit. The `try/finally` is what makes teardown unconditional — the generator
version of `__exit__`.

In [6]:
# Part 2 — the same manager via @contextmanager.
import contextlib

@contextlib.contextmanager
def database_connection(db_name):
    print(f"Opening {db_name} connection...")
    # try/finally mirrors __exit__: teardown always runs.
    try:
        yield  # the with block body runs here
    finally:
        print(f"Closing {db_name} connection...")

with database_connection("orders") as connection:
    print("Running a query inside the block...")

Opening orders connection...
Running a query inside the block...
Closing orders connection...


## Section 6 — Part 2: the same guarantee, fewer lines

Crash this version too. The `finally` block runs first, so the connection closes before
the error reaches the caller — identical behavior to the class version.

In [7]:
# The decorator version survives a crash too.
try:
    with database_connection("orders") as connection:
        raise RuntimeError("replica went offline")
except RuntimeError as error:
    print("Caught outside the block:", error)

Opening orders connection...
Closing orders connection...
Caught outside the block: replica went offline


## Section 7 — Part 3: a transaction that commits or rolls back

A transaction groups several statements into one unit: all succeed or all are undone.
`__exit__` tells you which happened — `exc_type is None` means the block succeeded, so
commit; anything else means roll back. The transaction is nested *inside* a connection.

In [8]:
# Part 3 — a transaction: commit on success, roll back on error.
class DatabaseTransaction:
    def __init__(self, connection):
        self.connection = connection
        self.statements = []  # the log, kept on commit, cleared on rollback

    def execute(self, statement):
        self.statements.append(statement)
        print("  executing:", statement)

    def __enter__(self):
        print(f"BEGIN transaction on {self.connection.db_name}")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:  # block succeeded -> commit
            print(f"COMMIT {len(self.statements)} statement(s)")
        else:
            print(f"ROLLBACK {len(self.statements)} statement(s)")
            self.statements.clear()  # discard the failed work
        return False

with DatabaseConnection("payments") as connection:
    with DatabaseTransaction(connection) as transaction:
        transaction.execute("INSERT INTO payments VALUES (1, 49.99)")
        transaction.execute("UPDATE accounts SET balance = balance - 49.99 WHERE id = 7")

print("Transaction log stored:", transaction.statements)

Opening payments connection...
BEGIN transaction on payments
  executing: INSERT INTO payments VALUES (1, 49.99)
  executing: UPDATE accounts SET balance = balance - 49.99 WHERE id = 7
COMMIT 2 statement(s)
Closing payments connection...
Transaction log stored: ['INSERT INTO payments VALUES (1, 49.99)', 'UPDATE accounts SET balance = balance - 49.99 WHERE id = 7']


## Section 8 — Part 3: the roll-back path

Same transaction, but the block raises mid-way. `__exit__` rolls back — clears the
statements, nothing committed — then re-raises so the caller can handle the error.

In [9]:
# The block raises, so __exit__ rolls the work back instead of committing.
try:
    with DatabaseConnection("payments") as connection:
        with DatabaseTransaction(connection) as transaction:
            transaction.execute("INSERT INTO payments VALUES (2, 199.99)")
            raise RuntimeError("payment gateway timed out")  # simulate a mid-transaction crash
except RuntimeError as error:
    print("Caught outside the block:", error)

print("Statements kept after rollback:", transaction.statements)

Opening payments connection...
BEGIN transaction on payments
  executing: INSERT INTO payments VALUES (2, 199.99)
ROLLBACK 1 statement(s)
Closing payments connection...
Caught outside the block: payment gateway timed out
Statements kept after rollback: []


## Section 9 — The vault ledger

Everything in this lab cleaned up after itself. The ledger below shows each resource and
its outcome in one glance.

In [10]:
from tabulate import tabulate

ledger = [
    ["vault_notes.txt", "write notes", "closed"],
    ["users connection", "SELECT + crash", "closed, error surfaced"],
    ["orders connection", "SELECT", "closed"],
    ["payments transaction", "2 statements", "committed"],
    ["payments transaction", "1 statement", "rolled back"],
]

print(tabulate(ledger, headers=["Resource", "Action", "Outcome"], tablefmt="grid"))

+----------------------+----------------+------------------------+
| Resource             | Action         | Outcome                |
+======================+================+========================+
| vault_notes.txt      | write notes    | closed                 |
+----------------------+----------------+------------------------+
| users connection     | SELECT + crash | closed, error surfaced |
+----------------------+----------------+------------------------+
| orders connection    | SELECT         | closed                 |
+----------------------+----------------+------------------------+
| payments transaction | 2 statements   | committed              |
+----------------------+----------------+------------------------+
| payments transaction | 1 statement    | rolled back            |
+----------------------+----------------+------------------------+
